In [35]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import os 
import pandas as pd

In [ ]:
torch.manual_seed(42)

In [ ]:
ROOT_DIR = os.path.abspath(os.path.join(os.path.join(os.getcwd(), ".."), ".."))
DATA_DIR = os.path.join(ROOT_DIR, "Data")
DATASET_DIR = os.path.join(DATA_DIR, "Fashion-MNIST")
TRAIN_DATASET_PATH = os.path.join(DATASET_DIR, "fashion-mnist_train.csv")
TEST_DATASET_PATH = os.path.join(DATASET_DIR, "fashion-mnist_test.csv")

In [37]:
train_data = pd.read_csv(TRAIN_DATASET_PATH)
test_data = pd.read_csv(TEST_DATASET_PATH)

X_train, y_train = train_data.drop("label", axis=1).values/255.0, train_data["label"].values
X_test, y_test = test_data.drop("label", axis=1).values/255.0, test_data["label"].values


In [38]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    

In [39]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [40]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [41]:
class NeuralNetworkClassifier(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, X):
        return self.model(X)

In [42]:
model1 = NeuralNetworkClassifier(num_features=X_train.shape[1], num_classes=len(set(y_train)))

In [43]:
Epochs = 100
learning_rate = 0.1

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model1.parameters(), lr=learning_rate)

In [44]:
for epoch in range(Epochs):
    total_loss = 0
    model1.train()
    for X_batch, y_batch in train_loader:
        y_pred = model1(X_batch)

        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    
    model1.eval()
    with torch.inference_mode():
        correct = 0
        total = 0
        for X_batch, y_batch in test_loader:
            y_pred = model1(X_batch)
            _, predicted = torch.max(y_pred.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
        
        accuracy = (correct / total) * 100
        print(f"Epoch [{epoch+1}/{Epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")


Epoch [1/100], Loss: 0.6024, Accuracy: 84.61%
Epoch [2/100], Loss: 0.4120, Accuracy: 84.35%
Epoch [3/100], Loss: 0.3680, Accuracy: 86.76%
Epoch [4/100], Loss: 0.3432, Accuracy: 87.24%
Epoch [5/100], Loss: 0.3239, Accuracy: 87.34%
Epoch [6/100], Loss: 0.3079, Accuracy: 88.05%
Epoch [7/100], Loss: 0.2972, Accuracy: 87.77%
Epoch [8/100], Loss: 0.2851, Accuracy: 88.50%
Epoch [9/100], Loss: 0.2777, Accuracy: 87.99%
Epoch [10/100], Loss: 0.2665, Accuracy: 87.93%
Epoch [11/100], Loss: 0.2609, Accuracy: 88.23%
Epoch [12/100], Loss: 0.2516, Accuracy: 88.99%
Epoch [13/100], Loss: 0.2438, Accuracy: 89.12%
Epoch [14/100], Loss: 0.2393, Accuracy: 87.86%
Epoch [15/100], Loss: 0.2332, Accuracy: 88.90%
Epoch [16/100], Loss: 0.2284, Accuracy: 89.42%
Epoch [17/100], Loss: 0.2236, Accuracy: 88.73%
Epoch [18/100], Loss: 0.2173, Accuracy: 88.88%
Epoch [19/100], Loss: 0.2143, Accuracy: 89.06%
Epoch [20/100], Loss: 0.2083, Accuracy: 88.62%
Epoch [21/100], Loss: 0.2027, Accuracy: 89.26%
Epoch [22/100], Loss: 